# Tierkreis assets and custom runtime configuration

This example runs a predefined graph with file-backed assets in a customized directory. Runtime configuration now owns asset storage, state, and executor setup.


In [ ]:
from pathlib import Path

from tierkreis import new_from_config

asset_dir = Path.home() / ".tierkreis" / "assets2"
config = {
    "asset_storage": {
        "memory": {"type": "Memory"},
        "file": {"type": "File", "asset_dir": str(asset_dir)},
    },
    "executors": {
        "subprocess": {
            "type": "Subprocess",
            "subprocess_storage_name": "file",
            "output_storage_name": "file",
        }
    },
    "runtime_state": {"type": "Memory"},
    "default_storage_name": "file",
    "default_executor_name": "subprocess",
    "logging_config": None,
}
runtime = await new_from_config(config)


The subprocess executor and file asset storage share this configuration. Installed worker commands are discovered on `PATH`.


In [ ]:
# No per-workflow storage or executor object is required.


Typically now one would define a graph.
Although, tierkreis ships some preconstructed graphs.
For this example we are going a to run a circuit on nexus and poll for its results.

In [ ]:
from tierkreis.graphs.nexus.submit_poll import nexus_submit_and_poll

graph = nexus_submit_and_poll()

Before we run the graph we can visualize it to get familiar with it.
This will start a webserver and listen on [localhost](http://localhost:8000)

In [ ]:
from qnexus.client.auth import login
from tierkreis_visualization.visualize_graph import visualize_graph

login()
visualize_graph(
    graph,
)  # this spawns a server, you need to manually terminate this cell.

Finally, provide JSON-compatible inputs and execute the graph through the configured runtime.


In [ ]:
from pytket.qasm.qasm import circuit_from_qasm
from qnexus import AerConfig

aer_config = AerConfig()
circuit = circuit_from_qasm(Path().parent / "data" / "ghz_state_n23.qasm")
inputs = {
    "project_name": "2025-tkr-test",
    "job_name": "job-1",
    "circuits": [circuit.to_dict()],
    "n_shots": [30],
    "backend_config": aer_config.model_dump(mode="json"),
}

with runtime:
    workflow_id = await runtime.save_workflow("nexus polling", graph)
    run_id = await runtime.start_new_run(workflow_id, inputs)
    await runtime.wait_for(run_id, 0)
    res = await runtime.get_outputs(run_id, 0)
